In [6]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

from scipy.signal import welch
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report, f1_score

from mhealth_activity.recording import Recording
from mhealth_activity.types import Activity

In [7]:
TRAIN_DIR = Path("data/train")
TEST_DIR = Path("data/test")

ACTIVITY_ORDER = ["standing", "walking", "running", "cycling"]

def parse_trace_id(path: Path) -> int:
    match = re.search(r"(\d+)\.pkl$", path.name)
    if match is None:
        raise ValueError(f"Could not parse Id from filename: {path.name}")
    return int(match.group(1))

def get_activity_names(labels):
    activity_ids = labels.get("activities", [])
    return [
        activity.name.lower()
        for activity in Activity
        if activity.value in activity_ids
    ]

Feature Extraction

In [8]:
def safe_trace_values(rec, key):
    if key not in rec.data:
        return None, None
    trace = rec.data[key]
    return trace.values.astype(float), float(trace.samplerate)

def basic_stats(x, prefix):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return {}

    return {
        f"{prefix}_mean": float(np.mean(x)),
        f"{prefix}_std": float(np.std(x)),
        f"{prefix}_min": float(np.min(x)),
        f"{prefix}_max": float(np.max(x)),
        f"{prefix}_range": float(np.max(x) - np.min(x)),
        f"{prefix}_rms": float(np.sqrt(np.mean(x**2))),
        f"{prefix}_q25": float(np.quantile(x, 0.25)),
        f"{prefix}_q50": float(np.quantile(x, 0.50)),
        f"{prefix}_q75": float(np.quantile(x, 0.75)),
        f"{prefix}_iqr": float(np.quantile(x, 0.75) - np.quantile(x, 0.25)),
    }

def spectral_features(x, fs, prefix):
    x = np.asarray(x, dtype=float)
    if len(x) < 16 or fs is None:
        return {
            f"{prefix}_dom_freq": 0.0,
            f"{prefix}_band_power_0_3": 0.0,
            f"{prefix}_band_power_3_8": 0.0,
            f"{prefix}_spec_entropy": 0.0,
            f"{prefix}_peak_sharpness": 0.0,
            f"{prefix}_dominance_ratio": 0.0,
        }

    x = x - np.mean(x)
    freqs, power = welch(x, fs=fs, nperseg=min(256, len(x)))

    total_power = np.sum(power) + 1e-12
    dom_freq = float(freqs[np.argmax(power)])

    band_0_3 = (freqs >= 0.3) & (freqs <= 3.0)
    band_3_8 = (freqs > 3.0) & (freqs <= 8.0)

    p = power / total_power
    spec_entropy = float(-(p * np.log(p + 1e-12)).sum())

    peak_power = float(np.max(power))
    mean_power = float(np.mean(power) + 1e-12)

    sorted_power = np.sort(power)
    top1 = float(sorted_power[-1])
    top2 = float(sorted_power[-2]) if len(sorted_power) >= 2 else 0.0

    return {
        f"{prefix}_dom_freq": dom_freq,
        f"{prefix}_band_power_0_3": float(np.sum(power[band_0_3])),
        f"{prefix}_band_power_3_8": float(np.sum(power[band_3_8])),
        f"{prefix}_spec_entropy": spec_entropy,
        f"{prefix}_peak_sharpness": peak_power / mean_power,
        f"{prefix}_dominance_ratio": top1 / (top2 + 1e-12),
    }


# testing if window-based features helps
def window_features(x, fs, prefix, window_s=10.0):
    x = np.asarray(x, dtype=float)
    if len(x) < 10 or fs is None:
        return {}

    win_len = int(window_s * fs)
    if win_len <= 0:
        return {}

    stds = []
    rms_vals = []
    ranges = []
    dom_freqs = []
    peak_sharpness_vals = []

    for start in range(0, len(x), win_len):
        seg = x[start:start + win_len]

        if len(seg) < max(10, win_len // 2):
            continue

        seg = seg.astype(float)
        seg_centered = seg - np.mean(seg)

        stds.append(np.std(seg_centered))
        rms_vals.append(np.sqrt(np.mean(seg_centered**2)))
        ranges.append(np.max(seg_centered) - np.min(seg_centered))

        if len(seg_centered) >= 16:
            freqs, power = welch(seg_centered, fs=fs, nperseg=min(128, len(seg_centered)))

            dom_freqs.append(float(freqs[np.argmax(power)]))

            peak_power = float(np.max(power))
            mean_power = float(np.mean(power) + 1e-12)
            peak_sharpness_vals.append(peak_power / mean_power)

    if len(stds) == 0:
        return {}

    out = {
        f"{prefix}_win_std_mean": float(np.mean(stds)),
        f"{prefix}_win_std_max": float(np.max(stds)),
        f"{prefix}_win_std_min": float(np.min(stds)),
        f"{prefix}_win_std_q75": float(np.quantile(stds, 0.75)),

        f"{prefix}_win_rms_mean": float(np.mean(rms_vals)),
        f"{prefix}_win_rms_max": float(np.max(rms_vals)),

        f"{prefix}_win_range_mean": float(np.mean(ranges)),
        f"{prefix}_win_range_max": float(np.max(ranges)),
    }

    if len(dom_freqs) > 0:
        out.update({
            f"{prefix}_win_dom_freq_mean": float(np.mean(dom_freqs)),
            f"{prefix}_win_dom_freq_std": float(np.std(dom_freqs)),
            f"{prefix}_win_dom_freq_min": float(np.min(dom_freqs)),
            f"{prefix}_win_dom_freq_max": float(np.max(dom_freqs)),
        })

    if len(peak_sharpness_vals) > 0:
        out.update({
            f"{prefix}_win_peak_sharpness_mean": float(np.mean(peak_sharpness_vals)),
            f"{prefix}_win_peak_sharpness_max": float(np.max(peak_sharpness_vals)),
            f"{prefix}_win_peak_sharpness_std": float(np.std(peak_sharpness_vals)),
        })

    return out

def extract_recording_features(rec):
    features = {}

    # smartwatch sensors only: always safer for Kaggle
    sensor_groups = {
        "acc": ["ax", "ay", "az"],
        "gyro": ["gx", "gy", "gz"],
        "mag": ["mx", "my", "mz"],
    }

    for group_name, keys in sensor_groups.items():
        vals = []
        fs_ref = None

        for key in keys:
            x, fs = safe_trace_values(rec, key)
            if x is None:
                continue

            vals.append(x)
            fs_ref = fs

            features.update(basic_stats(x, key))
            features.update(spectral_features(x, fs, key))

        # magnitude features
        if len(vals) == 3:
            min_len = min(len(v) for v in vals)
            x0, x1, x2 = [v[:min_len] for v in vals]
            mag = np.sqrt(x0**2 + x1**2 + x2**2)

            features.update(basic_stats(mag, f"{group_name}_mag"))
            features.update(spectral_features(mag, fs_ref, f"{group_name}_mag"))
            features.update(window_features(mag, fs_ref, f"{group_name}_mag")) #testing window base

            # axis correlations
            features[f"{group_name}_corr_xy"] = float(np.corrcoef(x0, x1)[0, 1]) if min_len > 2 else 0.0
            features[f"{group_name}_corr_xz"] = float(np.corrcoef(x0, x2)[0, 1]) if min_len > 2 else 0.0
            features[f"{group_name}_corr_yz"] = float(np.corrcoef(x1, x2)[0, 1]) if min_len > 2 else 0.0

    # duration
    if "ax" in rec.data:
        features["duration_s"] = float(rec.data["ax"].total_time)
    else:
        features["duration_s"] = 0.0

    return features

Build training table

In [9]:
rows = []

for path in sorted(TRAIN_DIR.glob("*.pkl")):
    rec = Recording(str(path))
    labels = rec.labels or {}

    activity_names = get_activity_names(labels)
    feat = extract_recording_features(rec)

    feat["file"] = path.name
    feat["trace_id"] = parse_trace_id(path)

    for activity in ACTIVITY_ORDER:
        feat[activity] = activity in activity_names

    rows.append(feat)

train_df = pd.DataFrame(rows).sort_values("trace_id").reset_index(drop=True)

print(train_df.shape)
train_df.head()

(396, 253)


,ax_mean,ax_std,ax_min,ax_max,ax_range,ax_rms,ax_q25,ax_q50,ax_q75,ax_iqr,...,mag_corr_xy,mag_corr_xz,mag_corr_yz,duration_s,file,trace_id,standing,walking,running,cycling
0,-0.296240,0.127974,-1.164795,0.562256,1.727051,0.322700,-0.373535,-0.284912,-0.199707,0.173828,...,0.533019,0.103512,-0.286065,576.042,train_trace_000.pkl,0,False,False,False,False
1,-0.002260,0.547424,-2.000000,1.999939,3.999939,0.547429,-0.309570,0.088379,0.333496,0.643066,...,0.198497,0.611224,-0.097020,788.925,train_trace_001.pkl,1,True,True,False,False
2,-0.074806,0.862987,-2.000000,1.999939,3.999939,0.866223,-0.173828,0.057373,0.220703,0.394531,...,-0.115788,0.150533,-0.013279,540.746,train_trace_002.pkl,2,False,True,False,False
3,0.802563,0.270869,-1.774170,1.999939,3.774109,0.847040,0.626221,0.760986,0.949219,0.322998,...,0.203943,-0.095999,-0.001194,733.139,train_trace_003.pkl,3,False,True,False,False
4,-0.850340,0.329182,-2.000000,0.074707,2.074707,0.911833,-1.089111,-0.865234,-0.598389,0.490723,...,-0.654207,0.250681,-0.008495,530.442,train_trace_004.pkl,4,False,True,False,False


Train 4 separate classifiers

In [10]:
target_cols = ACTIVITY_ORDER

drop_cols = ["file", "trace_id"] + target_cols
feature_cols = [c for c in train_df.columns if c not in drop_cols]

X = train_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
groups = train_df["trace_id"].values

classifiers = {}

for activity in target_cols:
    y = train_df[activity].astype(int).values

    clf = ExtraTreesClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )

    classifiers[activity] = clf

## Cross-validation ##

In [11]:
thresholds = {
    "standing": 0.25,
    "walking": 0.45,
    "running": 0.25,
    "cycling": 0.30,
}

cv = GroupKFold(n_splits=5)

for activity in target_cols:
    y = train_df[activity].astype(int).values
    oof_pred = np.zeros(len(train_df), dtype=int)

    for tr_idx, va_idx in cv.split(X, y, groups):
        clf = ExtraTreesClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )

        clf.fit(X.iloc[tr_idx], y[tr_idx])
        proba = clf.predict_proba(X.iloc[va_idx])[:, 1]
        oof_pred[va_idx] = (proba >= thresholds[activity]).astype(int)

    print("\n" + "=" * 50)
    print(activity)
    print("threshold:", thresholds[activity])
    print("F1:", f1_score(y, oof_pred, zero_division=0))
    print(classification_report(y, oof_pred, zero_division=0))


standing
threshold: 0.25
F1: 0.5853658536585366
              precision    recall  f1-score   support

           0       0.96      0.83      0.89       336
           1       0.46      0.80      0.59        60

    accuracy                           0.83       396
   macro avg       0.71      0.82      0.74       396
weighted avg       0.88      0.83      0.85       396


walking
threshold: 0.45
F1: 0.72
              precision    recall  f1-score   support

           0       0.60      0.26      0.36       169
           1       0.61      0.87      0.72       227

    accuracy                           0.61       396
   macro avg       0.61      0.57      0.54       396
weighted avg       0.61      0.61      0.57       396


running
threshold: 0.25
F1: 0.6031746031746031
              precision    recall  f1-score   support

           0       0.97      0.89      0.92       348
           1       0.49      0.79      0.60        48

    accuracy                           0.87       3

Out-of-fold validation
(every prediction is made by a model that did not train on that trace)

In [12]:
thresholds = {
    "standing": 0.25,
    "walking": 0.45,
    "running": 0.25,
    "cycling": 0.30,
}

cv = GroupKFold(n_splits=5)

val_pred_df = train_df[["file", "trace_id"] + target_cols].copy()

for activity in target_cols:
    y = train_df[activity].astype(int).values
    oof_proba = np.zeros(len(train_df), dtype=float)
    oof_pred = np.zeros(len(train_df), dtype=bool)

    for tr_idx, va_idx in cv.split(X, y, groups):
        clf = ExtraTreesClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )

        clf.fit(X.iloc[tr_idx], y[tr_idx])
        proba = clf.predict_proba(X.iloc[va_idx])[:, 1]

        oof_proba[va_idx] = proba
        oof_pred[va_idx] = proba >= thresholds[activity]

    val_pred_df[f"{activity}_proba"] = oof_proba
    val_pred_df[f"{activity}_pred"] = oof_pred
    val_pred_df[f"{activity}_correct"] = (
        val_pred_df[f"{activity}_pred"] == val_pred_df[activity]
    )

val_pred_df.head(20)

,file,trace_id,standing,walking,running,cycling,standing_proba,standing_pred,standing_correct,walking_proba,walking_pred,walking_correct,running_proba,running_pred,running_correct,cycling_proba,cycling_pred,cycling_correct
0,train_trace_000.pkl,0,False,False,False,False,0.312776,True,False,0.723125,True,False,0.006508,False,True,0.010966,False,True
1,train_trace_001.pkl,1,True,True,False,False,0.456298,True,True,0.468857,True,True,0.182233,False,True,0.123115,False,True
2,train_trace_002.pkl,2,False,True,False,False,0.012792,False,True,0.413761,False,False,0.064393,False,True,0.003609,False,True
3,train_trace_003.pkl,3,False,True,False,False,0.149437,False,True,0.530213,True,True,0.065136,False,True,0.163981,False,True
4,train_trace_004.pkl,4,False,True,False,False,0.041303,False,True,0.535418,True,True,0.054748,False,True,0.077528,False,True
5,train_trace_005.pkl,5,False,False,False,False,0.079899,False,True,0.514873,True,False,0.062471,False,True,0.027796,False,True
6,train_trace_006.pkl,6,False,False,False,False,0.045872,False,True,0.260623,False,True,0.166979,False,True,0.477141,True,False
7,train_trace_007.pkl,7,False,False,False,False,0.042492,False,True,0.550335,True,False,0.019024,False,True,0.027918,False,True
8,train_trace_008.pkl,8,False,True,True,False,0.253841,True,False,0.626337,True,True,0.420906,True,True,0.121170,False,True
9,train_trace_009.pkl,9,True,True,True,False,0.390613,True,True,0.589224,True,True,0.588162,True,True,0.079115,False,True


In [13]:
# fit final classifiers
for activity in target_cols:
    y = train_df[activity].astype(int).values
    classifiers[activity].fit(X, y)

print("Trained classifiers:", list(classifiers.keys()))

Trained classifiers: ['standing', 'walking', 'running', 'cycling']


Predicts activities for 1 recording

In [14]:
def predict_activities(rec, classifiers, feature_cols, thresholds=None):
    if thresholds is None:
        thresholds = {
            "standing": 0.25,
            "walking": 0.45,
            "running": 0.25,
            "cycling": 0.30,
        }

    feat = extract_recording_features(rec)
    x = pd.DataFrame([feat])

    # align columns exactly to training
    x = x.reindex(columns=feature_cols, fill_value=0.0)
    x = x.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    preds = {}

    for activity, clf in classifiers.items():
        proba = clf.predict_proba(x)[0, 1]
        preds[activity] = bool(proba >= thresholds[activity])

    return preds

Submission

In [15]:
test_dir = Path("data/test")
submission_name = "submission_activity.csv"

def parse_trace_id(path: Path) -> int:
    match = re.search(r"(\d+)\.pkl$", path.name)
    if match is None:
        raise ValueError(f"Could not parse Id from filename: {path.name}")
    return int(match.group(1))

rows = []

for path in sorted(test_dir.glob("*.pkl")):
    rec = Recording(str(path))

    activity_pred = predict_activities(
        rec,
        classifiers=classifiers,
        feature_cols=feature_cols,
        thresholds=thresholds,
    )

    rows.append({
        "Id": parse_trace_id(path),
        "watch_loc": -1,          # default
        "path_idx": -1,           # default
        "standing": activity_pred["standing"],
        "walking": activity_pred["walking"],
        "running": activity_pred["running"],
        "cycling": activity_pred["cycling"],
        "step_count": -1,         # default
    })

submission_df = pd.DataFrame(rows).sort_values("Id")

print(submission_df.head())
print(f"\nRows: {len(submission_df)}")
print("\nActivity counts:")
print(submission_df[["standing", "walking", "running", "cycling"]].sum())

submission_df.to_csv(submission_name, index=False)
print(f"\nSaved submission to {submission_name}")

   Id  watch_loc  path_idx  standing  walking  running  cycling  step_count
0   0         -1        -1      True     True     True    False          -1
1   1         -1        -1     False     True    False     True          -1
2   2         -1        -1      True     True    False     True          -1
3   3         -1        -1     False     True    False    False          -1
4   4         -1        -1     False     True    False    False          -1

Rows: 280

Activity counts:
standing     60
walking     228
running      53
cycling      30
dtype: int64

Saved submission to submission_activity.csv
